# Side-by-side PLY / MRI breathing motion viewer

Shows a PLY point cloud and an MRI slice side by side, each on its **own independent frame index**. Also plots the frame-to-frame motion amplitude of each modality, with a cursor per modality, and lets you set the PLY's viewing angle explicitly.

**Why the previous version didn't update**: it used `%matplotlib widget`, which needs the `ipympl` package installed and active. Without it, Jupyter silently keeps the static `inline` backend, so slider callbacks calling `fig.canvas.draw_idle()` have nothing to redraw. This version instead redraws into an `ipywidgets.Output()` box on every change, using the default `inline` backend -- this works with no extra package required, in Jupyter, JupyterLab, VS Code notebooks, and Colab alike.

**PLY viewing angle**: since each render is a fresh static image (not a live 3D canvas), free mouse-drag rotation isn't available here. Instead, use the `elev`/`azim` sliders below to set the viewing angle explicitly -- this is actually more reproducible for comparing the same angle across frames. If you'd rather have live mouse-drag 3D rotation and can get `ipympl` working in your environment (`pip install ipympl`, restart kernel, then swap `%matplotlib inline` for `%matplotlib widget` and remove the `Output()` wrapping around the PLY panel), that's a valid alternative -- ask if you'd like that version built out instead.

**Before running this**: `MRI_DIR` must point to a folder containing ONLY the volume-slice DICOM files (filtered via `explore_mri_series.py` + `split_dicom_by_attribute.py`, e.g. on `TriggerTime`), not navigator+volume mixed together.

**Units**: `PLY_UNIT_SCALE` converts your PLY coordinates to mm. Use `inspect_ply.py` first if you're unsure of your actual coordinate ranges -- the printed bounding-box size below is a sanity check, not a substitute for checking the real per-axis ranges.

In [ ]:
%matplotlib inline

import csv
from pathlib import Path

import numpy as np
import open3d as o3d
import pydicom
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import ipywidgets as widgets
from IPython.display import display, clear_output

## Config -- edit these paths for your data

In [ ]:
PLY_DIR = Path("/path/to/ply_folder")
MRI_DIR = Path("/path/to/mri_volume_slice_series_only")  # filtered, NOT the mixed navigator+volume folder

FRAME_INTERVAL_MS = 450.0     # nominal; real elapsed time is used instead when AcquisitionTime is available
PLY_UNIT_SCALE = 1000.0       # e.g. 1000.0 to convert metres -> mm; set to 1.0 if already mm
PLY_MAX_DISPLAY_POINTS = 3000 # subsample for responsive rendering (display only, not amplitude calc)

OUTPUT_CSV_PLY = Path("./output/ply_frame_to_frame_amplitude.csv")
OUTPUT_CSV_MRI = Path("./output/mri_frame_to_frame_amplitude.csv")

## Loading + temporal sorting helpers

In [ ]:
def looks_like_dicom(path: Path) -> bool:
    if not path.is_file():
        return False
    if path.name.upper() == "DICOMDIR":
        return False
    try:
        with open(path, "rb") as f:
            preamble = f.read(132)
    except OSError:
        return False
    return len(preamble) == 132 and preamble[128:132] == b"DICM"


def parse_dicom_time(time_str):
    if not time_str:
        return None
    time_str = time_str.strip()
    if len(time_str) < 6:
        return None
    try:
        return int(time_str[0:2]) * 3600 + int(time_str[2:4]) * 60 + float(time_str[4:])
    except ValueError:
        return None


def get_acquisition_time(path: Path):
    try:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
    except Exception:
        return None
    return parse_dicom_time(str(getattr(ds, "AcquisitionTime", "")))


def load_sorted_mri_files(mri_dir: Path):
    files = sorted(p for p in mri_dir.iterdir() if looks_like_dicom(p))
    if not files:
        raise ValueError(f"No DICOM files found in {mri_dir}")
    times = [get_acquisition_time(p) for p in files]
    if all(t is not None for t in times):
        order = sorted(range(len(files)), key=lambda i: times[i])
        files = [files[i] for i in order]
        times = [times[i] for i in order]
        t0 = times[0]
        time_ms = [(t - t0) * 1000.0 for t in times]
    else:
        print("  [warn] AcquisitionTime missing for some MRI frames -- "
              "falling back to filename order and a nominal frame interval.")
        time_ms = [i * FRAME_INTERVAL_MS for i in range(len(files))]
    return files, time_ms


def load_ply_points(path: Path, unit_scale: float) -> np.ndarray:
    pcd = o3d.io.read_point_cloud(str(path))
    pts = np.asarray(pcd.points, dtype=np.float64) * unit_scale
    if len(pts) == 0:
        raise ValueError(f"Empty point cloud: {path}")
    return pts


def subsample_deterministic(pts: np.ndarray, max_points: int, seed: int) -> np.ndarray:
    """Same seed (e.g. the frame index) -> same subset every time, so the
    displayed points don't randomly change between elev/azim adjustments."""
    if len(pts) <= max_points:
        return pts
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(pts), max_points, replace=False)
    return pts[idx]

## Load both sequences (independently -- lengths don't need to match)

In [ ]:
ply_files = sorted(PLY_DIR.glob("*.ply"))
mri_files, mri_time_ms = load_sorted_mri_files(MRI_DIR)

n_ply = len(ply_files)
n_mri = len(mri_files)
ply_time_ms = [i * FRAME_INTERVAL_MS for i in range(n_ply)]  # ply has no AcquisitionTime

print(f"PLY frames: {n_ply}")
print(f"MRI frames: {n_mri}")

sample_pts = load_ply_points(ply_files[0], PLY_UNIT_SCALE)
bbox_size = sample_pts.max(0) - sample_pts.min(0)
print(f"[units check] first PLY frame bbox size (x,y,z): {bbox_size} "
      f"(after PLY_UNIT_SCALE={PLY_UNIT_SCALE}) -- should be a few hundred mm "
      f"for a real abdominal surface if working in mm.")

## Precompute frame-to-frame amplitude for each modality independently

In [ ]:
def pointcloud_pair_amplitude(pts_a: np.ndarray, pts_b: np.ndarray) -> dict:
    tree_b = cKDTree(pts_b)
    d_a_to_b, _ = tree_b.query(pts_a, k=1, workers=-1)
    tree_a = cKDTree(pts_a)
    d_b_to_a, _ = tree_a.query(pts_b, k=1, workers=-1)
    d = np.concatenate([d_a_to_b, d_b_to_a])
    return {"mean": float(d.mean()), "rms": float(np.sqrt((d ** 2).mean())), "max": float(d.max())}


def mri_pair_amplitude(image_a: sitk.Image, image_b: sitk.Image) -> dict:
    a = sitk.Cast(image_a, sitk.sitkFloat32)
    b = sitk.Cast(image_b, sitk.sitkFloat32)
    b_resampled = sitk.Resample(b, a)

    demons = sitk.DemonsRegistrationFilter()
    demons.SetNumberOfIterations(50)
    demons.SetStandardDeviations(1.5)
    field = demons.Execute(a, b_resampled)

    field_array = sitk.GetArrayFromImage(field)
    magnitude = np.linalg.norm(field_array, axis=-1).ravel()
    return {"mean": float(magnitude.mean()), "rms": float(np.sqrt((magnitude ** 2).mean())),
            "max": float(magnitude.max())}


def compute_amplitude_series(n_frames, load_frame_fn, pair_amplitude_fn, time_ms_list, label):
    rows = []
    prev = None
    for i in range(n_frames):
        current = load_frame_fn(i)
        if i == 0:
            rows.append({"frame_index": i, "time_ms": time_ms_list[i],
                         "mean_mm": "", "rms_mm": "", "max_mm": ""})
        else:
            amp = pair_amplitude_fn(prev, current)
            rows.append({"frame_index": i, "time_ms": time_ms_list[i],
                         "mean_mm": amp["mean"], "rms_mm": amp["rms"], "max_mm": amp["max"]})
        prev = current
        if i % 20 == 0:
            print(f"  [{label}] processed {i + 1}/{n_frames}")
    return rows


ply_rows = compute_amplitude_series(
    n_ply,
    lambda i: load_ply_points(ply_files[i], PLY_UNIT_SCALE),
    pointcloud_pair_amplitude,
    ply_time_ms, "PLY",
)
mri_rows = compute_amplitude_series(
    n_mri,
    lambda i: sitk.ReadImage(str(mri_files[i])),
    mri_pair_amplitude,
    mri_time_ms, "MRI",
)

for path, rows in [(OUTPUT_CSV_PLY, ply_rows), (OUTPUT_CSV_MRI, mri_rows)]:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {path}")

ply_times_plot = [r["time_ms"] for r in ply_rows[1:]]
ply_rms_plot = [r["rms_mm"] for r in ply_rows[1:]]
mri_times_plot = [r["time_ms"] for r in mri_rows[1:]]
mri_rms_plot = [r["rms_mm"] for r in mri_rows[1:]]

## Interactive side-by-side viewer

Independent PLY/MRI frame sliders, plus `elev`/`azim` sliders to set the PLY viewing angle. Every control redraws the whole panel via `Output()` + `clear_output()` -- reliable across notebook environments, at the cost of live mouse-drag rotation.

In [ ]:
slider_ply = widgets.IntSlider(value=0, min=0, max=n_ply - 1, step=1, description="PLY frame")
btn_ply_prev = widgets.Button(description="< Prev")
btn_ply_next = widgets.Button(description="Next >")

slider_mri = widgets.IntSlider(value=0, min=0, max=n_mri - 1, step=1, description="MRI frame")
btn_mri_prev = widgets.Button(description="< Prev")
btn_mri_next = widgets.Button(description="Next >")

elev_slider = widgets.FloatSlider(value=20, min=-90, max=90, step=5, description="PLY elev")
azim_slider = widgets.FloatSlider(value=-60, min=0, max=360, step=5, description="PLY azim")

output_panel = widgets.Output()


def render_all(_=None):
    idx_ply = slider_ply.value
    idx_mri = slider_mri.value

    with output_panel:
        clear_output(wait=True)

        fig = plt.figure(figsize=(12, 8))
        ax_ply = fig.add_subplot(2, 2, 1, projection="3d")
        ax_mri = fig.add_subplot(2, 2, 2)
        ax_curve = fig.add_subplot(2, 1, 2)

        pts = load_ply_points(ply_files[idx_ply], PLY_UNIT_SCALE)
        pts = subsample_deterministic(pts, PLY_MAX_DISPLAY_POINTS, seed=idx_ply)
        ax_ply.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1, c=pts[:, 2], cmap="viridis")
        ax_ply.set_title(f"External surface (PLY)  frame {idx_ply}/{n_ply - 1}  "
                         f"t={ply_time_ms[idx_ply]:.0f} ms")
        ax_ply.set_box_aspect([1, 1, 1])
        ax_ply.view_init(elev=elev_slider.value, azim=azim_slider.value)

        mri_img = sitk.ReadImage(str(mri_files[idx_mri]))
        mri_array = sitk.GetArrayFromImage(mri_img)
        if mri_array.ndim == 3:
            mri_array = mri_array[0]
        vmin, vmax = np.percentile(mri_array, [1, 99])
        ax_mri.imshow(mri_array, cmap="gray", vmin=vmin, vmax=vmax)
        ax_mri.set_title(f"Internal slice (MRI)  frame {idx_mri}/{n_mri - 1}  "
                         f"t={mri_time_ms[idx_mri]:.0f} ms")
        ax_mri.axis("off")

        ax_curve.plot(ply_times_plot, ply_rms_plot, label="PLY surface (RMS, mm)", color="tab:blue")
        ax_curve.plot(mri_times_plot, mri_rms_plot, label="MRI slice, in-plane (RMS, mm)", color="tab:orange")
        ax_curve.axvline(ply_time_ms[idx_ply], color="tab:blue", linestyle="--", alpha=0.7)
        ax_curve.axvline(mri_time_ms[idx_mri], color="tab:orange", linestyle="--", alpha=0.7)
        ax_curve.set_xlabel("time (ms)")
        ax_curve.set_ylabel("amplitude (mm)")
        ax_curve.legend()

        plt.tight_layout()
        plt.show()


for w in (slider_ply, slider_mri, elev_slider, azim_slider):
    w.observe(render_all, names="value")
btn_ply_prev.on_click(lambda _: setattr(slider_ply, "value", max(0, slider_ply.value - 1)))
btn_ply_next.on_click(lambda _: setattr(slider_ply, "value", min(n_ply - 1, slider_ply.value + 1)))
btn_mri_prev.on_click(lambda _: setattr(slider_mri, "value", max(0, slider_mri.value - 1)))
btn_mri_next.on_click(lambda _: setattr(slider_mri, "value", min(n_mri - 1, slider_mri.value + 1)))

controls = widgets.VBox([
    widgets.HBox([btn_ply_prev, slider_ply, btn_ply_next]),
    widgets.HBox([btn_mri_prev, slider_mri, btn_mri_next]),
    widgets.HBox([elev_slider, azim_slider]),
])

render_all()
display(controls, output_panel)

## Notes

- Amplitude values are frame-to-frame (t vs. t+1 within each modality's own sequence), not amplitude-from-a-fixed-reference.
- If a slider still doesn't visibly update anything: check the cell above actually re-ran (`render_all()` should print no errors), and that you're not running this in an environment that caches Output widget state oddly (restart kernel + Run All as a first troubleshooting step).
- `PLY_MAX_DISPLAY_POINTS` subsampling uses `subsample_deterministic`, seeded by frame index -- so the same points display every time you view a given frame, and only the elev/azim/frame changes actually change what you see (no distracting per-render randomness).